# 1. Preprocesamiento de datos — Deteccion de fraude (PaySim)

Carga el dataset crudo `PS_20174392719_1491204439457_log.csv`, aplica feature
engineering y genera el split temporal pedido por el proyecto:

- **Referencia / entrenamiento**: `step <= 400`
- **Produccion (simulacion online)**: `step > 400`

Ver diccionario de datos completo en [`data/README.md`](../data/README.md)
y en el `README.md` principal del repo.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.config import (
    PRODUCTION_DATA_PATH,
    PROCESSED_DIR,
    RAW_DATA_PATH,
    REFERENCE_DATA_PATH,
    REFERENCE_MAX_STEP,
)
from src.features import build_dummies, engineer_features

pd.set_option("display.max_columns", None)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Leyendo raw data desde:", RAW_DATA_PATH)

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)
df.head()

## 1.1 Exploracion rapida

In [ ]:
df.info()
df.isna().sum()

In [ ]:
print("Distribucion de la clase objetivo (isFraud):")
print(df["isFraud"].value_counts(normalize=True))
print()
print("Distribucion por tipo de transaccion:")
print(df["type"].value_counts())
print()
print("Tasa de fraude por tipo de transaccion:")
print(df.groupby("type")["isFraud"].mean().sort_values(ascending=False))

Nota de negocio (Anexo 1): el fraude en PaySim solo ocurre en
transacciones `TRANSFER` y `CASH_OUT` (el agente toma control de la cuenta,
transfiere el dinero y luego lo retira). Lo validamos a continuacion.

In [ ]:
fraud_by_type = df.loc[df["isFraud"] == 1, "type"].value_counts()
print(fraud_by_type)
assert set(fraud_by_type.index) <= {"TRANSFER", "CASH_OUT"}, "Fraude fuera de TRANSFER/CASH_OUT

## 1.2 Feature engineering

Ver `src/features.py` para el detalle de cada feature derivada.

In [ ]:
df_feat = engineer_features(df)
df_feat = build_dummies(df_feat)
df_feat.filter(regex="error|Ratio|empt|Merchant|Unknown|^type_").head()

## 1.3 Split temporal: referencia vs produccion

`step <= 400` -> referencia (entrenamiento/validacion offline). `step > 400` -> produccion (simulacion de monitoreo online).

In [ ]:
reference = df_feat[df_feat["step"] <= REFERENCE_MAX_STEP].reset_index(drop=True)
production = df_feat[df_feat["step"] > REFERENCE_MAX_STEP].reset_index(drop=True)

print(f"Referencia : {reference.shape} | tasa de fraude = {reference['isFraud'].mean():.5%}")
print(f"Produccion : {production.shape} | tasa de fraude = {production['isFraud'].mean():.5%}")

In [ ]:
reference.to_csv(REFERENCE_DATA_PATH, index=False)
production.to_csv(PRODUCTION_DATA_PATH, index=False)
print("Guardado:")
print(" -", REFERENCE_DATA_PATH)
print(" -", PRODUCTION_DATA_PATH)

## 1.4 Siguiente paso

Continuar con [`2.0-entrenamiento-modelo.ipynb`](2.0-entrenamiento-modelo.ipynb)
para el entrenamiento, optimizacion de hiperparametros y registro en MLflow.